# Recommendations

Synthesise all prior analyses into a prioritised action list with build time savings estimates and ownership mapping.

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = Path("../../data/processed")
PROJECT_ROOT = Path("../..")
INTERMEDIATE_DIR = Path("../../data/intermediate")

from buildanalysis.loading import BuildDataset

ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)

plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100, "axes.titlesize": 14})
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
# Load core data
target_metrics = ds.target_metrics
edge_list = ds.edge_list
contributor_target = ds.contributor_target_commits

# Optional intermediates from prior notebooks
ownership = None
ownership_path = INTERMEDIATE_DIR / "target_ownership.parquet"
if ownership_path.exists():
    ownership = pd.read_parquet(ownership_path)
    print(f"Loaded target_ownership: {len(ownership)} targets, columns: {list(ownership.columns)}")

pch_candidates_df = None
pch_path = INTERMEDIATE_DIR / "pch_candidates.parquet"
if pch_path.exists():
    pch_candidates_df = pd.read_parquet(pch_path)
    print(f"Loaded pch_candidates: {len(pch_candidates_df)} candidates")

module_metrics = None
module_path = INTERMEDIATE_DIR / "module_metrics.parquet"
if module_path.exists():
    module_metrics = pd.read_parquet(module_path)
    print(f"Loaded module_metrics: {len(module_metrics)} modules, columns: {list(module_metrics.columns)}")

module_assignments = None
module_assignments_path = INTERMEDIATE_DIR / "module_assignments.parquet"
if module_assignments_path.exists():
    module_assignments = pd.read_parquet(module_assignments_path)
    print(f"Loaded module_assignments: {len(module_assignments)} targets")

feature_configs = None
feature_configs_path = INTERMEDIATE_DIR / "feature_configurations.parquet"
if feature_configs_path.exists():
    feature_configs = pd.read_parquet(feature_configs_path)
    print(f"Loaded feature_configurations: {len(feature_configs)} modules")

centrality_data = None
centrality_path = INTERMEDIATE_DIR / "centrality_metrics.parquet"
if centrality_path.exists():
    centrality_data = pd.read_parquet(centrality_path)
    print(f"Loaded centrality: {len(centrality_data)} targets, columns: {list(centrality_data.columns)}")

transitive_deps = None
transitive_deps_path = INTERMEDIATE_DIR / "transitive_deps.parquet"
if transitive_deps_path.exists():
    transitive_deps = pd.read_parquet(transitive_deps_path)
    print(f"Loaded transitive_deps: {len(transitive_deps)} targets")

layers_data = None
layers_path = INTERMEDIATE_DIR / "layer_assignments.parquet"
if layers_path.exists():
    layers_data = pd.read_parquet(layers_path)
    print(f"Loaded layer_assignments: {len(layers_data)} targets")

communities_data = None
communities_path = INTERMEDIATE_DIR / "community_assignments.parquet"
if communities_path.exists():
    communities_data = pd.read_parquet(communities_path)
    print(f"Loaded community_assignments: {len(communities_data)} targets")

print(f"\nCore data loaded: {len(target_metrics)} targets, {len(edge_list)} edges")

## 1. Critical Path Targets

In [ ]:
# Build graph and compute critical path using library functions
from buildanalysis.graph import build_dependency_graph
from buildanalysis.build import compute_critical_path

bg = build_dependency_graph(target_metrics, edge_list)

timing = target_metrics[["cmake_target", "total_build_time_ms"]].copy()
try:
    cp = compute_critical_path(bg, timing)
    critical_path_time_ms = cp.total_time_s * 1000
    critical_path_targets_set = set(cp.path)
    print(f"Critical path length: ~{critical_path_time_ms:.0f} ms ({len(cp.path)} targets)")
except Exception as e:
    print(f"Warning: Could not compute critical path: {e}")
    cp = None
    critical_path_time_ms = 0
    critical_path_targets_set = set()

# Use betweenness centrality from NB06 if available, otherwise recompute
if centrality_data is not None and "betweenness" in centrality_data.columns:
    betweenness_map = centrality_data.set_index("cmake_target")["betweenness"].to_dict()
else:
    import networkx as nx
    betweenness_map = nx.betweenness_centrality(bg.graph)

critical_targets = []
for _, tdata in target_metrics.iterrows():
    target = tdata["cmake_target"]
    build_time = tdata["total_build_time_ms"] if pd.notna(tdata["total_build_time_ms"]) else 0
    bet = betweenness_map.get(target, 0.0)
    
    # Get owner from ownership data (uses standard column names: cmake_target, owning_team)
    owner = "unknown"
    if ownership is not None and "cmake_target" in ownership.columns and "owning_team" in ownership.columns:
        owner_rows = ownership[ownership["cmake_target"] == target]
        if len(owner_rows) > 0:
            ot = owner_rows.iloc[0]["owning_team"]
            owner = str(ot) if pd.notna(ot) else "unknown"
    
    critical_targets.append({
        "target": target,
        "target_type": tdata["target_type"],
        "build_time_ms": build_time,
        "betweenness_centrality": bet,
        "on_critical_path": target in critical_path_targets_set,
        "owner": owner,
        "priority_score": bet * np.log1p(build_time),
    })

critical_df = pd.DataFrame(critical_targets).sort_values("priority_score", ascending=False)

print("\nCRITICAL PATH TARGETS (Top 15 by priority score)")
print("=" * 90)
display_cols = ["target", "target_type", "build_time_ms", "betweenness_centrality", "on_critical_path", "owner"]
print(critical_df[display_cols].head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 8))
top_critical = critical_df.head(15)
ax.barh(range(len(top_critical)), top_critical["priority_score"])
ax.set_yticks(range(len(top_critical)))
ax.set_yticklabels(top_critical["target"], fontsize=9)
ax.set_xlabel("Priority Score (betweenness x log(build_time))")
ax.set_title("Top 15 Critical Path Targets")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 2. PCH Recommendations

In [ ]:
if pch_candidates_df is not None and len(pch_candidates_df) > 0:
    pch_df = pch_candidates_df.copy()
    
    # Use the 'score' column from nb05 output
    sort_col = "score" if "score" in pch_df.columns else "estimated_time_savings_ms"
    if sort_col not in pch_df.columns:
        pch_df["score"] = 0
        sort_col = "score"
    
    pch_df = pch_df.sort_values(sort_col, ascending=False)
    
    print("\nPRECOMPILED HEADER (PCH) RECOMMENDATIONS (Top 10)")
    print("=" * 90)
    display_cols = [col for col in ["header", "target", "score"] if col in pch_df.columns]
    print(pch_df[display_cols].head(10).to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(12, 6))
    top_pch = pch_df.head(10)
    ax.barh(range(len(top_pch)), top_pch[sort_col].values)
    ax.set_yticks(range(len(top_pch)))
    labels = top_pch["header"].values if "header" in top_pch.columns else [f"PCH {i}" for i in range(len(top_pch))]
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel(f"PCH Score")
    ax.set_title("Top 10 PCH Candidates")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No PCH candidates available.")

## 3. Dependency Reduction

In [ ]:
# Use transitive_dep_count from target_metrics (pre-computed) or NB06 intermediate
if transitive_deps is not None:
    dep_counts_df = transitive_deps.merge(
        target_metrics[["cmake_target", "target_type", "total_build_time_ms"]],
        on="cmake_target", how="left"
    ).rename(columns={"n_transitive_deps": "transitive_dep_count", "total_build_time_ms": "build_time_ms"})
elif "transitive_dependency_count" in target_metrics.columns:
    dep_counts_df = target_metrics[["cmake_target", "target_type", "transitive_dependency_count", "total_build_time_ms"]].copy()
    dep_counts_df = dep_counts_df.rename(columns={"transitive_dependency_count": "transitive_dep_count", "total_build_time_ms": "build_time_ms"})
else:
    # Fallback: compute from graph
    import networkx as nx
    dep_rows = []
    for target in target_metrics["cmake_target"].unique():
        if target in bg.graph:
            n_deps = len(nx.descendants(bg.graph, target))
        else:
            n_deps = 0
        tdata = target_metrics[target_metrics["cmake_target"] == target].iloc[0]
        dep_rows.append({
            "cmake_target": target,
            "target_type": tdata["target_type"],
            "transitive_dep_count": n_deps,
            "build_time_ms": tdata["total_build_time_ms"],
        })
    dep_counts_df = pd.DataFrame(dep_rows)

dep_counts_df = dep_counts_df.sort_values("transitive_dep_count", ascending=False)

print("\nDEPENDENCY REDUCTION OPPORTUNITIES (Top 15 by transitive deps)")
print("=" * 80)
print(dep_counts_df[["cmake_target", "target_type", "transitive_dep_count", "build_time_ms"]].head(15).to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
top_deps = dep_counts_df.head(20)
ax.barh(range(len(top_deps)), top_deps["transitive_dep_count"])
ax.set_yticks(range(len(top_deps)))
ax.set_yticklabels(top_deps["cmake_target"], fontsize=9)
ax.set_xlabel("Transitive Dependency Count")
ax.set_title("Top 20 Targets by Transitive Dependencies")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Module Health

In [ ]:
if module_metrics is not None and len(module_metrics) > 0:
    print("\nMODULE HEALTH FLAGS")
    print("=" * 80)
    
    # Use self_containment (0-1 scale from compute_module_metrics)
    if "self_containment" in module_metrics.columns:
        low_containment = module_metrics[module_metrics["self_containment"] < 0.5]
        if len(low_containment) > 0:
            print(f"\n{len(low_containment)} module(s) with <50% self-containment:")
            for _, row in low_containment.iterrows():
                print(f"  - {row['module']}: {row['self_containment']*100:.0f}% self-contained")
        else:
            print("  All modules above 50% self-containment threshold.")
        
        # Show critical path and build fraction if available
        if "critical_path_target_count" in module_metrics.columns:
            cp_modules = module_metrics[module_metrics["critical_path_target_count"] > 0]
            if len(cp_modules) > 0:
                print(f"\n{len(cp_modules)} module(s) with targets on critical path:")
                for _, row in cp_modules.iterrows():
                    print(f"  - {row['module']}: {int(row['critical_path_target_count'])} target(s)")
    
    display_cols = [c for c in ["module", "category", "target_count", "self_containment",
                                 "total_build_time_ms", "build_fraction", "critical_path_target_count"]
                    if c in module_metrics.columns]
    print(f"\nModule metrics summary:")
    print(module_metrics[display_cols].to_string(index=False))
else:
    print("No module metrics available.")

## 5. Ownership Risks

In [ ]:
if ownership is not None:
    # Detect the target column name (standard: cmake_target)
    target_col = "cmake_target" if "cmake_target" in ownership.columns else "target"

    # Use ownership_hhi as a proxy for ownership concentration risk
    if "ownership_hhi" in ownership.columns:
        high_concentration = ownership[ownership["ownership_hhi"] > 0.8]

        critical_set = set(critical_df["target"].head(20))
        at_risk = high_concentration[high_concentration[target_col].isin(critical_set)]

        print("\nOWNERSHIP RISKS")
        print("=" * 80)
        print(f"Targets with high ownership concentration (HHI > 0.8): {len(high_concentration)}")
        print(f"Of those, in top-20 critical targets: {len(at_risk)}")
        if len(at_risk) > 0:
            display_cols = [c for c in [target_col, "owning_team", "ownership_hhi", "contributor_count", "cross_team_fraction"]
                           if c in at_risk.columns]
            print("\nAt-Risk Targets:")
            print(at_risk[display_cols].to_string(index=False))
    elif "bus_factor" in ownership.columns:
        single_owner = ownership[ownership["bus_factor"] == 1]
        critical_set = set(critical_df["target"].head(20))
        at_risk = single_owner[single_owner[target_col].isin(critical_set)]

        print("\nOWNERSHIP RISKS")
        print("=" * 80)
        print(f"Targets with bus factor = 1: {len(single_owner)}")
        print(f"Of those, in top-20 critical: {len(at_risk)}")
        if len(at_risk) > 0:
            print("\nAt-Risk Targets:")
            print(at_risk.to_string(index=False))
    else:
        print("Ownership data does not contain expected columns.")

    # File-level hotspot detail from NB02's file_churn intermediate
    file_churn_path = INTERMEDIATE_DIR / "file_churn.parquet"
    if file_churn_path.exists() and len(at_risk) > 0:
        from buildanalysis.git import compute_file_to_target_map
        file_churn = pd.read_parquet(file_churn_path)
        file_metrics_fm = ds.file_metrics
        file_to_target = compute_file_to_target_map(file_metrics_fm)

        # Show hotspot files within at-risk targets
        at_risk_targets = set(at_risk[target_col])
        hotspot_files = file_churn[file_churn["source_file"].isin(file_to_target.index)]
        hotspot_files = hotspot_files.copy()
        hotspot_files["cmake_target"] = hotspot_files["source_file"].map(file_to_target)
        hotspot_files = hotspot_files[hotspot_files["cmake_target"].isin(at_risk_targets)]

        if len(hotspot_files) > 0:
            print(f"\nFile-Level Hotspots in At-Risk Targets (top 15):")
            for _, row in hotspot_files.nlargest(15, "n_commits").iterrows():
                print(f"  {row['source_file']:60s} {row['n_commits']:4d} commits ({row['cmake_target']})")
else:
    print("No ownership data available.")

## 6. Prioritised Action List

In [ ]:
# Use library scoring functions to generate proper Intervention objects
from buildanalysis.recommend import (
    Intervention, InterventionType, build_pareto_frontier,
    score_header_interventions, score_dependency_interventions,
    score_target_split_interventions, format_recommendation_summary,
)

all_interventions: list[Intervention] = []

# 1. Target split interventions (critical path targets with many files)
if cp is not None:
    split_ivs = score_target_split_interventions(target_metrics, cp, top_n=10)
    all_interventions.extend(split_ivs)
    print(f"Target split interventions: {len(split_ivs)}")

# 2. Dependency removal interventions
if cp is not None:
    dep_ivs = score_dependency_interventions(bg, timing, cp, edge_list, top_n=20)
    all_interventions.extend(dep_ivs)
    print(f"Dependency interventions: {len(dep_ivs)}")

# 3. Header refactoring interventions (from NB05 PCH analysis)
header_impact_path = INTERMEDIATE_DIR / "header_impact.parquet"
if header_impact_path.exists():
    header_impact = pd.read_parquet(header_impact_path)
    amplification = pd.DataFrame(columns=["file", "direct_includes", "transitive_includes", "amplification_ratio"])
    header_ivs = score_header_interventions(header_impact, amplification, top_n=20)
    all_interventions.extend(header_ivs)
    print(f"Header interventions: {len(header_ivs)}")

# 4. Module self-containment interventions
if module_metrics is not None and "self_containment" in module_metrics.columns:
    for _, row in module_metrics[module_metrics["self_containment"] < 0.5].head(5).iterrows():
        mod_name = row["module"]
        # Resolve owning team from module config or metrics
        team = str(row.get("owning_team", "architecture")) if pd.notna(row.get("owning_team")) else "architecture"
        all_interventions.append(Intervention(
            intervention_type=InterventionType.EXTRACT_INTERFACE,
            description=f"Improve self-containment of module {mod_name}",
            targets_affected=[],
            estimated_build_time_reduction_ms=0,
            estimated_effort_days=5.0,
            confidence=0.3,
            team=team,
            module=mod_name,
            rationale=f"Module {mod_name} has {row['self_containment']*100:.0f}% self-containment, below 50% threshold.",
        ))

print(f"\nTotal interventions generated: {len(all_interventions)}")

# Build Pareto frontier with all 11 attribute-reference columns
rec_df = build_pareto_frontier(all_interventions)

print("\nPRIORITISED ACTION LIST")
print("=" * 100)
display_cols = [c for c in ["type", "description", "impact_ms", "effort_days", "confidence",
                             "category", "team", "module", "pareto_optimal"] if c in rec_df.columns]
print(rec_df[display_cols].to_string(index=False))

# Save with all 11 columns per attribute reference
ds.save_intermediate("recommendations", rec_df)
print(f"\nSaved {len(rec_df)} recommendations ({len(rec_df.columns)} columns)")

# Summary
if len(rec_df) > 0:
    total_estimated_savings = rec_df["impact_ms"].sum()
    pareto_count = rec_df["pareto_optimal"].sum()
    print(f"\nTotal estimated savings: {total_estimated_savings:.0f} ms (~{total_estimated_savings/60000:.1f} minutes)")
    print(f"Pareto-optimal recommendations: {pareto_count}")
    
    print("\n" + format_recommendation_summary(rec_df, top_n=10))

## 7. Export for Gephi

In [ ]:
# Export all 4 GEXF graphs with full attribute sets per attribute_reference.md
from buildanalysis.export import export_dependency_graph, export_module_graph, export_include_graph, export_cochange_graph

output_dir = Path("../../data/intermediate/gephi")
output_dir.mkdir(parents=True, exist_ok=True)

# --- 1. Dependency Graph GEXF (29 node attrs, 7 edge attrs) ---
if centrality_data is not None and layers_data is not None and communities_data is not None:
    cent_for_export = centrality_data.copy()
    if transitive_deps is not None and "n_transitive_deps" in transitive_deps.columns:
        cent_for_export = cent_for_export.merge(
            transitive_deps[["cmake_target", "n_transitive_deps"]], on="cmake_target", how="left"
        )
    
    dep_path = export_dependency_graph(
        bg=bg, centrality=cent_for_export, layers=layers_data,
        communities=communities_data, timing=timing,
        critical_path_result=cp, target_ownership=ownership,
        module_assignments=module_assignments,
        output_path=output_dir / "dependency_graph.gexf",
    )
    print(f"Exported dependency graph: {dep_path}")
else:
    print("Skipping dependency graph export (missing centrality, layers, or communities from NB06)")

# --- 2. Module Graph GEXF (13 node attrs, 5 edge attrs) ---
modules_path = PROJECT_ROOT / "modules.yaml"
if modules_path.exists() and module_metrics is not None and module_assignments is not None:
    from buildanalysis.modules import ModuleConfig, build_module_dependency_graph
    mod_config = ModuleConfig.from_yaml(modules_path)
    mod_graph = build_module_dependency_graph(bg, module_assignments)
    mod_path = export_module_graph(
        module_graph=mod_graph, module_config=mod_config,
        module_metrics=module_metrics, feature_configs=feature_configs,
        output_path=output_dir / "module_graph.gexf",
    )
    print(f"Exported module graph: {mod_path}")
else:
    print("Skipping module graph export (missing config, metrics, or assignments)")

# --- 3. Include Graph GEXF (21 node attrs, 5 edge attrs) ---
if ds.has_file("header_edges"):
    from buildanalysis.graph import build_include_graph
    from buildanalysis.headers import (
        compute_include_fan_metrics, compute_header_impact_score,
        compute_header_pagerank, compute_include_amplification,
    )
    header_edges = ds.header_edges
    header_metrics_df = ds.header_metrics if ds.has_file("header_metrics") else None
    include_graph = build_include_graph(header_edges)
    fan_metrics = compute_include_fan_metrics(include_graph)
    file_metrics = ds.file_metrics
    git_churn_df = file_metrics[["source_file", "git_commit_count", "git_churn"]].rename(
        columns={"git_commit_count": "n_commits", "git_churn": "total_churn"}
    ) if "git_commit_count" in file_metrics.columns else pd.DataFrame(columns=["source_file", "n_commits", "total_churn"])
    
    if header_metrics_df is not None:
        header_impact_df = compute_header_impact_score(fan_metrics, header_metrics_df, git_churn_df)
    else:
        header_impact_df = fan_metrics.copy()
        header_impact_df["impact_score"] = 0.0
    header_pr = compute_header_pagerank(include_graph)
    amplification_df = compute_include_amplification(include_graph, file_metrics)
    
    inc_path = export_include_graph(
        include_graph=include_graph,
        header_metrics=header_metrics_df if header_metrics_df is not None else pd.DataFrame(
            columns=["header_file", "cmake_target", "sloc", "source_size_bytes", "is_system"]
        ),
        header_impact=header_impact_df, header_pagerank=header_pr,
        git_churn=git_churn_df, file_metrics=file_metrics,
        module_assignments=module_assignments, team_ownership=ownership,
        amplification=amplification_df, exclude_system=True,
        output_path=output_dir / "include_graph.gexf",
    )
    print(f"Exported include graph: {inc_path}")
    ds.save_intermediate("header_impact", header_impact_df)
else:
    print("Skipping include graph export (no header_edges data)")

# --- 4. Co-change Graph GEXF (12 node attrs, 8 edge attrs) ---
if ds.has_file("git_commit_log"):
    from buildanalysis.git import compute_cochange_matrix
    git_log = ds.git_commit_log
    file_metrics = ds.file_metrics
    # Deduplicate source_file before creating the index
    file_to_target = (
        file_metrics.drop_duplicates("source_file")
        .set_index("source_file")["cmake_target"]
    ) if "source_file" in file_metrics.columns else None
    
    try:
        cochange = compute_cochange_matrix(git_log, file_to_target=file_to_target, level="target", min_cochanges=3)
        if len(cochange) > 0 and communities_data is not None:
            git_churn_target = target_metrics[["cmake_target", "git_commit_count_total", "git_churn_total", "git_distinct_authors"]].rename(
                columns={"git_commit_count_total": "n_commits", "git_churn_total": "total_churn", "git_distinct_authors": "contributor_count"}
            ) if "git_commit_count_total" in target_metrics.columns else pd.DataFrame(columns=["cmake_target", "n_commits", "total_churn"])
            
            cc_path = export_cochange_graph(
                cochange=cochange, target_metrics=target_metrics,
                git_churn=git_churn_target, structural_communities=communities_data,
                edge_list=edge_list, module_assignments=module_assignments,
                team_ownership=ownership, min_pmi=0.0,
                output_path=output_dir / "cochange_graph.gexf",
            )
            print(f"Exported co-change graph: {cc_path}")
        else:
            print("Skipping co-change graph (insufficient co-change data or missing communities)")
    except Exception as e:
        print(f"Could not export co-change graph: {e}")
else:
    print("Skipping co-change graph export (no git_commit_log data)")